# SGBA

Combined Colab/Kaggle notebook for the SGBA revision experiments.
Run on either Google Colab (auto-mounts Drive) or Kaggle (auto-detects dataset).

In [ ]:
# Setup: paths, device, and training constants

import os
import sys
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime

# Try Colab first, then Kaggle
try:
    from google.colab import drive
    drive.mount('/content/drive')
    INPUT_ROOT = Path('/content/drive/MyDrive/SGBA_Revision')
    OUTPUT_ROOT = INPUT_ROOT
    print(f"Colab mode: {INPUT_ROOT}")
except:
    # Kaggle
    kaggle_paths = [
        Path('/kaggle/input/datasets/mahmudurlabib/sgba-revision-2/SGBA_Revision'),
        Path('/kaggle/input/sgba-revision/SGBA_Revision'),
    ]
    INPUT_ROOT = None
    for p in kaggle_paths:
        if p.exists():
            INPUT_ROOT = p
            break
    if INPUT_ROOT is None:
        INPUT_ROOT = Path('./SGBA_Revision')
    OUTPUT_ROOT = Path('/kaggle/working/SGBA_Revision') if Path('/kaggle/working').exists() else INPUT_ROOT
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Kaggle/local mode: {INPUT_ROOT}")

DKASC_DIR = INPUT_ROOT / 'data' / 'processed' / 'dkasc'
SOLAR_DIR = INPUT_ROOT / 'data' / 'solar_position'
INPUT_METRICS = INPUT_ROOT / 'results' / 'metrics'
INPUT_CKPTS = INPUT_ROOT / 'models' / 'checkpoints'
INPUT_PRED_CACHE = INPUT_ROOT / 'results' / 'predictions'

OUTPUT_METRICS = OUTPUT_ROOT / 'results' / 'metrics'
OUTPUT_CKPTS = OUTPUT_ROOT / 'models' / 'checkpoints'
OUTPUT_PRED_CACHE = OUTPUT_ROOT / 'results' / 'predictions'
OUTPUT_CURVES = OUTPUT_ROOT / 'results' / 'training_curves'
OUTPUT_PLOTS = OUTPUT_ROOT / 'results' / 'plots'

for d in [OUTPUT_METRICS, OUTPUT_CKPTS, OUTPUT_PRED_CACHE, OUTPUT_CURVES, OUTPUT_PLOTS]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = [42, 123, 7]
EPOCHS = 100
BATCH_SIZE = 128
LR = 1e-3
WD = 1e-5
GRAD_CLIP = 1.0

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU only")
print(f"PyTorch: {torch.__version__}")
print(f"Python: {sys.version.split()[0]}")


## 1. Model Definitions

SGBA, SGBABlock, AnomalyExtractor, SGBABlockBase, and SGBAControl.

In [ ]:
from __future__ import annotations

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class AnomalyExtractor(nn.Module):
    """Fixed EWMA envelope extraction with warm-start initialization."""
    def __init__(self, alpha=0.9):
        super().__init__()
        self.register_buffer('alpha', torch.tensor(float(alpha)))

    def forward(self, x):
        B, L, C = x.shape
        envelope = torch.zeros_like(x)
        env = x[:, 0, :].clone()  # warm-start
        for t in range(L):
            env = self.alpha * env + (1.0 - self.alpha) * x[:, t, :]
            envelope[:, t, :] = env
        return x - envelope, envelope


class SGBABlock(nn.Module):
    """Attention block with bilinear solar geometry bias.

    B[i,j] = g_i^T M_h g_j. M_h is zero-initialized so SGBA reduces to
    standard attention at init.
    """
    def __init__(self, d_model=96, n_heads=4, geo_dim=32, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        # Bilinear bias: B[i,j] = g_i^T M_h g_j for each head h
        self.geo_bias_M = nn.Parameter(
            torch.zeros(n_heads, geo_dim, geo_dim)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model * 4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.d_head ** -0.5

    def forward(self, tokens, geo_features):
        B, L, _ = tokens.shape
        H, Dh = self.n_heads, self.d_head

        Q = self.W_q(tokens).view(B, L, H, Dh).transpose(1, 2)
        K = self.W_k(tokens).view(B, L, H, Dh).transpose(1, 2)
        V = self.W_v(tokens).view(B, L, H, Dh).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if geo_features is not None:
            proj = torch.einsum('blg,hgj->bhlj', geo_features, self.geo_bias_M)
            bias = torch.einsum('bhig,bjg->bhij', proj, geo_features)
            scores = scores + bias

        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        out = self.W_o(out)

        h = self.norm1(tokens + self.dropout(out))
        h = self.norm2(h + self.dropout(self.ffn(h)))
        return h, attn


class SGBA(nn.Module):
    """Solar Geometry-Biased Attention.

    Attention head only. Zero-init final layer so SGBA starts with
    zero predictions.
    """
    def __init__(self, seq_len=24, n_features=5, d_model=96, n_heads=4,
                 n_blocks=2, geo_dim=32, dropout=0.1,
                 geometry_mode='ewma',
                 ghi_envelope_mean=0.0, ghi_envelope_std=1.0,
                 elev_mean=0.0, elev_std=1.0):
        super().__init__()
        self.seq_len = seq_len
        self.power_idx = 0
        self.ghi_idx = 3
        self.geometry_mode = geometry_mode
        self.geo_dim = geo_dim

        self.register_buffer('ghi_envelope_mean', torch.tensor(float(ghi_envelope_mean)))
        self.register_buffer('ghi_envelope_std', torch.tensor(float(ghi_envelope_std)))
        self.register_buffer('elev_mean', torch.tensor(float(elev_mean)))
        self.register_buffer('elev_std', torch.tensor(float(elev_std)))

        if geometry_mode == 'ewma':
            self.anomaly_extractor = AnomalyExtractor(alpha=0.9)

        self.input_proj = nn.Linear(n_features, d_model)

        # Nonlinear geo projection: Linear -> ReLU -> Linear
        self.geo_proj = nn.Sequential(
            nn.Linear(4, geo_dim), nn.ReLU(), nn.Linear(geo_dim, geo_dim)
        )

        self.blocks = nn.ModuleList([
            SGBABlock(d_model, n_heads, geo_dim, dropout)
            for _ in range(n_blocks)
        ])

        # Attention head only
        self.attn_head = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model, 1)
        )
        nn.init.zeros_(self.attn_head[-1].weight)
        nn.init.zeros_(self.attn_head[-1].bias)

    def compute_geo_features(self, x, hour, elevation=None, envelope=None):
        B, L, _ = x.shape
        # hour-of-day from timestamp, not window index
        pos_sin = torch.sin(2 * math.pi * hour / 24.0)
        pos_cos = torch.cos(2 * math.pi * hour / 24.0)

        if self.geometry_mode == 'ewma':
            if envelope is None:
                _, envelope = self.anomaly_extractor(x)
            geo_signal = envelope[:, :, self.ghi_idx]
            # global normalization with training-set stats
            geo_signal_norm = (geo_signal - self.ghi_envelope_mean) / (self.ghi_envelope_std + 1e-6)
        else:  # 'elev'
            geo_signal = elevation
            geo_signal_norm = (geo_signal - self.elev_mean) / (self.elev_std + 1e-6)

        geo_raw = torch.stack([pos_sin, pos_cos, geo_signal_norm, geo_signal_norm ** 2], dim=-1)
        return self.geo_proj(geo_raw)

    def forward(self, x, hour, elevation=None):
        envelope = None
        if self.geometry_mode == 'ewma':
            _, envelope = self.anomaly_extractor(x)
        geo_features = self.compute_geo_features(x, hour, elevation, envelope)
        h = self.input_proj(x)
        for block in self.blocks:
            h, _ = block(h, geo_features)
        attn_repr = torch.cat([h[:, -1, :], h.mean(dim=1)], dim=-1)
        return self.attn_head(attn_repr)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class SGBABlockBase(nn.Module):
    """Base attention block supporting different bias types for ablation controls."""
    def __init__(self, d_model=96, n_heads=4, geo_dim=32, dropout=0.1, bias_type='bilinear'):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.bias_type = bias_type

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        if bias_type in ('bilinear', 'sin_cos_only', 'elev_only', 'shuffled'):
            self.geo_bias_M = nn.Parameter(torch.zeros(n_heads, geo_dim, geo_dim))
        elif bias_type == 'alibi':
            self.alibi_slope = nn.Parameter(torch.zeros(n_heads))

        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model * 4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.d_head ** -0.5

    def forward(self, tokens, geo_features, hour=None):
        B, L, _ = tokens.shape
        H, Dh = self.n_heads, self.d_head

        Q = self.W_q(tokens).view(B, L, H, Dh).transpose(1, 2)
        K = self.W_k(tokens).view(B, L, H, Dh).transpose(1, 2)
        V = self.W_v(tokens).view(B, L, H, Dh).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if self.bias_type in ('bilinear', 'sin_cos_only', 'elev_only', 'shuffled'):
            proj = torch.einsum('blg,hgj->bhlj', geo_features, self.geo_bias_M)
            bias = torch.einsum('bhig,bjg->bhij', proj, geo_features)
            scores = scores + bias
        elif self.bias_type == 'alibi':
            positions = torch.arange(L, device=tokens.device).float()
            dist = torch.abs(positions.unsqueeze(0) - positions.unsqueeze(1))
            bias = -self.alibi_slope.view(1, H, 1, 1) * dist.unsqueeze(0).unsqueeze(0)
            scores = scores + bias

        attn = self.dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        out = self.W_o(out)

        h = self.norm1(tokens + self.dropout(out))
        h = self.norm2(h + self.dropout(self.ffn(h)))
        return h, attn


class SGBAControl(nn.Module):
    """SGBA variant for ablation controls.

    bias_type options:
        'bilinear'      : full SGBA-Elev (headline)
        'sin_cos_only'  : bias uses only sin/cos (no elevation)
        'elev_only'     : bias uses only elevation (no sin/cos)
        'shuffled'      : bias uses fixed-permutation elevation (null control)
        'alibi'         : ALiBi distance bias (no geometry)
        'none'          : no bias (standard attention)

    pe_type options (added to tokens, independent of bias):
        None            : no positional encoding on tokens
        'sinusoidal'    : standard sinusoidal PE added to tokens
        'learned'       : learned PE added to tokens

    Note: 'shuffled' uses a fixed permutation set at init time
    (deterministic), instead of random per forward pass. This makes
    evaluation reproducible. The permutation varies by global seed.
    """
    def __init__(self, seq_len=24, n_features=5, d_model=96, n_heads=4,
                 n_blocks=2, geo_dim=32, dropout=0.1,
                 bias_type='bilinear', pe_type=None,
                 ghi_envelope_mean=0.0, ghi_envelope_std=1.0,
                 elev_mean=0.0, elev_std=1.0):
        super().__init__()
        self.seq_len = seq_len
        self.power_idx = 0
        self.ghi_idx = 3
        self.bias_type = bias_type
        self.pe_type = pe_type
        self.geo_dim = geo_dim

        self.register_buffer('ghi_envelope_mean', torch.tensor(float(ghi_envelope_mean)))
        self.register_buffer('ghi_envelope_std', torch.tensor(float(ghi_envelope_std)))
        self.register_buffer('elev_mean', torch.tensor(float(elev_mean)))
        self.register_buffer('elev_std', torch.tensor(float(elev_std)))

        self.input_proj = nn.Linear(n_features, d_model)

        # Positional encoding on tokens (independent of bias)
        if pe_type == 'sinusoidal':
            pe = torch.zeros(seq_len, d_model)
            position = torch.arange(0, seq_len).unsqueeze(1).float()
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer('pe_sinusoidal', pe.unsqueeze(0))
        elif pe_type == 'learned':
            self.pe_learned = nn.Parameter(torch.zeros(1, seq_len, d_model))
            nn.init.normal_(self.pe_learned, std=0.02)

        # Geo projection (only for bias types that use geo_features)
        if bias_type in ('bilinear', 'sin_cos_only', 'elev_only', 'shuffled'):
            self.geo_proj = nn.Sequential(
                nn.Linear(4, geo_dim), nn.ReLU(), nn.Linear(geo_dim, geo_dim)
            )

        # Fixed permutation for shuffled bias (set at init, deterministic)
        if bias_type == 'shuffled':
            perm = torch.randperm(seq_len)
            self.register_buffer('shuffle_perm', perm)

        self.blocks = nn.ModuleList([
            SGBABlockBase(d_model, n_heads, geo_dim, dropout, bias_type=bias_type)
            for _ in range(n_blocks)
        ])

        self.attn_head = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model, 1)
        )
        nn.init.zeros_(self.attn_head[-1].weight)
        nn.init.zeros_(self.attn_head[-1].bias)

    def compute_geo_features(self, x, hour, elevation):
        B, L, _ = x.shape
        pos_sin = torch.sin(2 * math.pi * hour / 24.0)
        pos_cos = torch.cos(2 * math.pi * hour / 24.0)
        elev_norm = (elevation - self.elev_mean) / (self.elev_std + 1e-6)

        if self.bias_type == 'bilinear':
            geo_raw = torch.stack([pos_sin, pos_cos, elev_norm, elev_norm ** 2], dim=-1)
        elif self.bias_type == 'sin_cos_only':
            zeros = torch.zeros_like(pos_sin)
            geo_raw = torch.stack([pos_sin, pos_cos, zeros, zeros], dim=-1)
        elif self.bias_type == 'elev_only':
            zeros = torch.zeros_like(pos_sin)
            geo_raw = torch.stack([zeros, zeros, elev_norm, elev_norm ** 2], dim=-1)
        elif self.bias_type == 'shuffled':
            # Fixed permutation (deterministic) — null control
            elev_shuffled = elev_norm[:, self.shuffle_perm]
            geo_raw = torch.stack([pos_sin, pos_cos, elev_shuffled, elev_shuffled ** 2], dim=-1)
        else:
            geo_raw = torch.zeros(B, L, 4, device=x.device)

        return self.geo_proj(geo_raw)

    def forward(self, x, hour, elevation=None):
        h = self.input_proj(x)

        if self.pe_type == 'sinusoidal':
            h = h + self.pe_sinusoidal
        elif self.pe_type == 'learned':
            h = h + self.pe_learned

        geo_features = None
        if self.bias_type in ('bilinear', 'sin_cos_only', 'elev_only', 'shuffled'):
            geo_features = self.compute_geo_features(x, hour, elevation)

        for block in self.blocks:
            h, _ = block(h, geo_features, hour=hour)

        attn_repr = torch.cat([h[:, -1, :], h.mean(dim=1)], dim=-1)
        return self.attn_head(attn_repr)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


## 2. Data Loading

PVDatasetV2, create_dataloaders_v2, and compute_metrics.

In [ ]:
import json
import random
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


# Standard channel ordering used across all datasets.
CHANNEL_NAMES = ["power", "temp", "humidity", "ghi", "dhi"]


class PVDatasetV2(Dataset):
    """PV forecasting dataset returning (x, y, hour, elevation).

    Reads pre-processed .npy files from a site directory.
    """
    def __init__(self, site_dir, split='train'):
        self.site_dir = Path(site_dir)
        self.X = np.load(self.site_dir / f'X_{split}.npy')
        self.y = np.load(self.site_dir / f'y_{split}.npy')
        self.hour = np.load(self.site_dir / f'hour_{split}.npy')
        self.elev = np.load(self.site_dir / f'elev_{split}.npy')

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32),
            torch.tensor(self.hour[idx], dtype=torch.float32),
            torch.tensor(self.elev[idx], dtype=torch.float32),
        )


def create_dataloaders_v2(site_key, data_root, batch_size=128, num_workers=2, seed=42):
    """Create train/val/test loaders for a DKASC or PVGIS site.

    Args:
        site_key: 'site7', 'site10', 'site2', 'singapore', 'oslo', 'madrid'
        data_root: Path to the data root (e.g. 'data/processed')
        batch_size: batch size
        num_workers: number of DataLoader workers
        seed: random seed for shuffling

    Returns:
        train_loader, val_loader, test_loader, norm_stats
        norm_stats = {
            'ghi_envelope_mean': float,
            'ghi_envelope_std': float,
            'elev_mean': float,
            'elev_std': float,
        }
    """
    data_root = Path(data_root)

    # Determine site directory (DKASC sites are under dkasc/, PVGIS under pvgis_)
    if site_key.startswith('site'):
        site_dir = data_root / 'dkasc' / site_key
    else:
        site_dir = data_root / f'pvgis_{site_key}'

    def seed_worker(worker_id):
        np.random.seed(seed + worker_id)
        random.seed(seed + worker_id)

    train_ds = PVDatasetV2(site_dir, 'train')
    val_ds = PVDatasetV2(site_dir, 'val')
    test_ds = PVDatasetV2(site_dir, 'test')

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True,
                              worker_init_fn=seed_worker, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    # Load normalization stats from configs/geometry_normalization_stats.json
    norm_path = data_root.parent / 'configs' / 'geometry_normalization_stats.json'
    if not norm_path.exists():
        norm_path = data_root / 'configs' / 'geometry_normalization_stats.json'

    if norm_path.exists():
        with open(norm_path) as f:
            norm_data = json.load(f)

        # Handle both site7 and dkasc_site7 key formats
        elev_key = site_key if site_key in norm_data.get('elevation_stats', {}) else f'dkasc_{site_key}'
        ghi_key = site_key if site_key in norm_data.get('ghi_envelope_stats', {}) else f'dkasc_{site_key}'

        norm_stats = {
            'ghi_envelope_mean': norm_data['ghi_envelope_stats'].get(ghi_key, {}).get('mean', 0.0),
            'ghi_envelope_std': norm_data['ghi_envelope_stats'].get(ghi_key, {}).get('std', 1.0),
            'elev_mean': norm_data['elevation_stats'].get(elev_key, {}).get('mean', 0.0),
            'elev_std': norm_data['elevation_stats'].get(elev_key, {}).get('std', 1.0),
        }
    else:
        # Fallback: compute from training data
        print(f"Warning: geometry_normalization_stats.json not found at {norm_path}")
        print(f"  Computing stats from training data elevation...")
        elev_train = np.load(site_dir / 'elev_train.npy')
        norm_stats = {
            'ghi_envelope_mean': 0.0,
            'ghi_envelope_std': 1.0,
            'elev_mean': float(elev_train.mean()),
            'elev_std': float(elev_train.std() + 1e-8),
        }

    return train_loader, val_loader, test_loader, norm_stats


def load_site_normalization(site_key, data_root):
    """Load normalization_params.json for a site.

    Returns:
        dict with 'means' and 'stds' for each channel.
    """
    data_root = Path(data_root)
    if site_key.startswith('site'):
        site_dir = data_root / 'dkasc' / site_key
    else:
        site_dir = data_root / f'pvgis_{site_key}'

    norm_path = site_dir / 'normalization_params.json'
    if norm_path.exists():
        with open(norm_path) as f:
            return json.load(f)
    return None


def compute_metrics(y_true_norm, y_pred_norm, site_key, data_root):
    """All metrics in original kW scale using site-specific normalization.

    Args:
        y_true_norm: normalized true values
        y_pred_norm: normalized predicted values
        site_key: site identifier (e.g. 'site7')
        data_root: Path to processed data root

    Returns:
        dict with RMSE, MAE, R2, MBE, n
    """
    import math

    norm = load_site_normalization(site_key, data_root)
    if norm is None:
        # Fallback: return normalized metrics
        y_true = np.asarray(y_true_norm).flatten()
        y_pred = np.asarray(y_pred_norm).flatten()
    else:
        pm, ps = norm['means']['power'], norm['stds']['power']
        y_true = np.asarray(y_true_norm) * ps + pm
        y_pred = np.asarray(y_pred_norm) * ps + pm
        y_true = y_true.flatten()
        y_pred = y_pred.flatten()

    rmse = math.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    mbe = np.mean(y_pred - y_true)

    return {'RMSE': float(rmse), 'MAE': float(mae), 'R2': float(r2),
            'MBE': float(mbe), 'n': int(len(y_true))}


## 3. Training & Evaluation

set_seed, train_sgba_control, evaluate_sgba_control.

In [ ]:
import os
import time
import math
import json
import random
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


def set_seed(seed: int = 42) -> None:
    """Set seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def train_sgba_control(model, train_loader, val_loader, *,
                      epochs=EPOCHS, lr=LR, weight_decay=WD,
                      grad_clip=GRAD_CLIP, device=DEVICE,
                      checkpoint_path=None, verbose=False, exp_id=None):
    """Train an SGBAControl (or SGBA) model. Returns history dict."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}
    best_val = float('inf')
    best_state = None

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        losses = []
        for x_b, y_b, hour_b, elev_b in train_loader:
            x_b = x_b.to(device, non_blocking=True)
            y_b = y_b.to(device, non_blocking=True)
            hour_b = hour_b.to(device, non_blocking=True)
            elev_b = elev_b.to(device, non_blocking=True)
            pred = model(x_b, hour_b, elev_b)
            optimizer.zero_grad()
            loss = criterion(pred.squeeze(-1), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            losses.append(loss.item())

        avg_tr = float(np.mean(losses))

        model.eval()
        vlosses = []
        with torch.no_grad():
            for x_b, y_b, hour_b, elev_b in val_loader:
                x_b = x_b.to(device, non_blocking=True)
                y_b = y_b.to(device, non_blocking=True)
                hour_b = hour_b.to(device, non_blocking=True)
                elev_b = elev_b.to(device, non_blocking=True)
                pred = model(x_b, hour_b, elev_b)
                vlosses.append(criterion(pred.squeeze(-1), y_b).item())

        avg_va = float(np.mean(vlosses))
        history['train_loss'].append(avg_tr)
        history['val_loss'].append(avg_va)
        scheduler.step(avg_va)

        if avg_va < best_val:
            best_val = avg_va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if checkpoint_path is not None:
                torch.save({'model_state': best_state, 'epoch': epoch,
                            'val_loss': best_val, 'exp_id': exp_id}, checkpoint_path)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == epochs - 1):
            elapsed = time.time() - t0
            print(f"  Epoch {epoch+1:3d}/{epochs}: tr={avg_tr:.5f} va={avg_va:.5f} "
                  f"best={best_val:.5f} ({elapsed:.0f}s)")

    if best_state is not None:
        model.load_state_dict(best_state)

    history['best_val_loss'] = best_val
    history['train_time'] = time.time() - t0
    return {'model': model, 'history': history,
            'best_val_loss': best_val,
            'n_params': sum(p.numel() for p in model.parameters())}


def evaluate_sgba_control(model, loader, *, device=DEVICE,
                          site_key=None, data_root=None):
    """Evaluate the model on a loader. Returns metrics + predictions."""
    model = model.to(device)
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for x_b, y_b, hour_b, elev_b in loader:
            x_b = x_b.to(device)
            y_b = y_b.to(device)
            hour_b = hour_b.to(device)
            elev_b = elev_b.to(device)
            p = model(x_b, hour_b, elev_b)
            preds.append(p.squeeze(-1).cpu().numpy())
            trues.append(y_b.cpu().numpy())

    yp = np.concatenate(preds)
    yt = np.concatenate(trues)

    metrics = {}
    if site_key is not None and data_root is not None:
        metrics = compute_metrics(yt, yp, site_key, data_root)

    return {'metrics': metrics, 'y_pred_norm': yp, 'y_true_norm': yt}


## 4. Baseline Models

PatchTST, CNNModule, CTNet, SimpleMLP, SimpleLSTM, SimpleGRU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PatchTST(nn.Module):
    """Simplified PatchTST for univariate forecasting."""
    def __init__(self, seq_len=24, pred_len=1, patch_len=4, stride=4,
                 d_model=64, n_heads=4, n_layers=2, dropout=0.1, power_idx=0):
        super().__init__()
        self.power_idx = power_idx
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1
        self.patch_embedding = nn.Linear(patch_len, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.normal_(self.pos_embedding, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x, hour=None, elevation=None):
        x_power = x[:, :, self.power_idx]  # (B, L) — univariate
        patches = x_power.unfold(1, self.patch_len, self.stride)  # (B, n_patches, patch_len)
        embedded = self.patch_embedding(patches) + self.pos_embedding  # (B, n_patches, d_model)
        encoded = self.encoder(embedded)  # (B, n_patches, d_model)
        flattened = encoded.flatten(1)  # (B, n_patches * d_model)
        out = self.output_proj(flattened)  # (B, 1)
        return out

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class CNNModule(nn.Module):
    def __init__(self, n_features=5, n_filters=12):
        super().__init__()
        self.conv1 = nn.Conv1d(n_features, n_filters, 3, padding=1)
        self.conv2 = nn.Conv1d(n_filters, n_filters, 3, padding=1)
        self.conv3 = nn.Conv1d(n_filters, n_filters, 3, padding=1)
        self.projection = nn.Conv1d(n_filters, n_filters, 1)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x)); x = F.relu(self.conv2(x)); x = F.relu(self.conv3(x))
        return self.projection(x).transpose(1, 2)


class CTNet(nn.Module):
    """CNN-Transformer hybrid (Munsif et al. 2023)."""
    def __init__(self, n_features=5, d_model=12, n_heads=4, n_encoder_blocks=3,
                 encoder_dropout=0.2, decoder_dropout=0.3):
        super().__init__()
        class EB(nn.Module):
            def __init__(s, nf, dm, nh, dp):
                super().__init__()
                s.cnn = CNNModule(nf, dm)
                s.mha = nn.MultiheadAttention(dm, nh, dropout=dp, batch_first=True)
                s.norm1 = nn.LayerNorm(dm)
                s.ffn = nn.Sequential(nn.Linear(dm, dm*4), nn.ReLU(),
                                      nn.Dropout(dp), nn.Linear(dm*4, dm))
                s.norm2 = nn.LayerNorm(dm); s.dp = nn.Dropout(dp)
            def forward(s, x):
                c = s.cnn(x); a, _ = s.mha(c, c, c)
                x = s.norm1(c + s.dp(a)); return s.norm2(x + s.dp(s.ffn(x)))
        class Dec(nn.Module):
            def __init__(s, dm, nh, dp):
                super().__init__()
                s.mha = nn.MultiheadAttention(dm, nh, dropout=dp, batch_first=True)
                s.ffd = nn.Sequential(nn.Linear(dm, 32), nn.ReLU(), nn.Dropout(dp),
                                      nn.Linear(32, 64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 1))
            def forward(s, x):
                a, _ = s.mha(x, x, x); return s.ffd((x + a).mean(dim=1))
        self.encs = nn.ModuleList([EB(n_features if i==0 else d_model, d_model, n_heads, encoder_dropout)
                                    for i in range(n_encoder_blocks)])
        self.dec = Dec(d_model, n_heads, decoder_dropout)
    def forward(self, x, hour=None, elevation=None):
        for b in self.encs: x = b(x)
        return self.dec(x)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class SimpleMLP(nn.Module):
    def __init__(self, input_dim=5, seq_len=24, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim * seq_len, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Linear(hidden_dim // 2, 1))
    def forward(self, x, hour=None, elevation=None):
        return self.net(x.flatten(1))

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class SimpleLSTM(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, n_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x, hour=None, elevation=None):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class SimpleGRU(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, n_layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x, hour=None, elevation=None):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


## 5. Multi-Horizon

MultiHorizonDataset, build_multihorizon_targets, train_multihorizon, evaluate_multihorizon.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class MultiHorizonDataset(Dataset):
    """Dataset for multi-horizon forecasting. Reuses existing X arrays with custom y targets."""
    def __init__(self, X, y_h, hour, elev):
        self.X = X
        self.y = y_h
        self.hour = hour
        self.elev = elev

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32),
            torch.tensor(self.hour[idx], dtype=torch.float32),
            torch.tensor(self.elev[idx], dtype=torch.float32),
        )


def build_multihorizon_targets(ts, y, horizon):
    """Build h-step-ahead targets by timestamp lookup.

    For window i with h=1 target at ts[i]:
      h-step target = y[j] where ts[j] = ts[i] + (horizon - 1) hours

    Windows where the target timestamp doesn't exist (nighttime gap
    or out of range) are dropped.

    Returns: (y_h, valid_mask) where valid_mask[i]=True means window i
    has a valid h-step target.
    """
    import pandas as pd
    ts_pd = pd.to_datetime(ts)
    ts_to_idx = {}
    for i, t in enumerate(ts_pd):
        ts_to_idx[t] = i

    y_h = np.full(len(y), np.nan, dtype=np.float32)
    for i, t in enumerate(ts_pd):
        target_ts = t + pd.Timedelta(hours=horizon - 1)
        if target_ts in ts_to_idx:
            j = ts_to_idx[target_ts]
            y_h[i] = y[j]

    valid_mask = ~np.isnan(y_h)
    return y_h, valid_mask


def _train_one(model, train_loader, val_loader, *, epochs=EPOCHS, lr=LR,
               weight_decay=WD, grad_clip=GRAD_CLIP, device=DEVICE,
               use_geometry=True, verbose=False):
    """Generic training loop matching train_sgba_control, supports baselines."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_val = float('inf')
    best_state = None

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch in train_loader:
            x_b, y_b = batch[0].to(device), batch[1].to(device)
            if use_geometry:
                hour_b, elev_b = batch[2].to(device), batch[3].to(device)
                pred = model(x_b, hour_b, elev_b)
            else:
                pred = model(x_b)
            optimizer.zero_grad()
            loss = criterion(pred.squeeze(-1), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            losses.append(loss.item())

        model.eval()
        vlosses = []
        with torch.no_grad():
            for batch in val_loader:
                x_b, y_b = batch[0].to(device), batch[1].to(device)
                if use_geometry:
                    hour_b, elev_b = batch[2].to(device), batch[3].to(device)
                    pred = model(x_b, hour_b, elev_b)
                else:
                    pred = model(x_b)
                vlosses.append(criterion(pred.squeeze(-1), y_b).item())

        avg_va = float(np.mean(vlosses))
        scheduler.step(avg_va)
        if avg_va < best_val:
            best_val = avg_va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if verbose and ((epoch + 1) % 10 == 0 or epoch == epochs - 1):
            print(f"  Epoch {epoch+1:3d}/{epochs}: val={avg_va:.5f} best={best_val:.5f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def train_multihorizon(model, train_loader, val_loader, *, use_geometry=True,
                        epochs=EPOCHS, lr=LR, weight_decay=WD, grad_clip=GRAD_CLIP,
                        device=DEVICE, verbose=False):
    """Train a model on a multi-horizon loader. Returns the trained model."""
    return _train_one(model, train_loader, val_loader, epochs=epochs, lr=lr,
                      weight_decay=weight_decay, grad_clip=grad_clip,
                      device=device, use_geometry=use_geometry, verbose=verbose)


def evaluate_multihorizon(model, loader, *, use_geometry=True, device=DEVICE,
                          site_key=None, data_root=None):
    """Evaluate model on a multi-horizon loader. Returns metrics dict."""
    model = model.to(device)
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            x_b, y_b = batch[0].to(device), batch[1].to(device)
            if use_geometry:
                hour_b, elev_b = batch[2].to(device), batch[3].to(device)
                p = model(x_b, hour_b, elev_b)
            else:
                p = model(x_b)
            preds.append(p.squeeze(-1).cpu().numpy())
            trues.append(y_b.cpu().numpy())

    yp = np.concatenate(preds)
    yt = np.concatenate(trues)

    metrics = {}
    if site_key is not None and data_root is not None:
        metrics = compute_metrics(yt, yp, site_key, data_root)
    return {'metrics': metrics, 'y_pred_norm': yp, 'y_true_norm': yt}


## 6. Run Ablation Experiments

SGBA-Elev (bilinear) vs SGBA-ShuffledElev across 3 seeds and 3 sites.

In [ ]:
import json
import numpy as np
import torch

ABLATION_SITES = ['site7', 'site10', 'site2']
DATA_ROOT = DKASC_DIR.parent  # .../data/processed

ablation_results = {}

for site in ABLATION_SITES:
    train_loader, val_loader, test_loader, norm_stats = create_dataloaders_v2(
        site, DATA_ROOT, batch_size=BATCH_SIZE, num_workers=2, seed=42)
    print(f"\n{site}: train={len(train_loader.dataset)} "
          f"val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

    for bias_type, label in [('bilinear', 'SGBA-Elev'),
                             ('shuffled', 'SGBA-ShuffledElev')]:
        ablation_results.setdefault(site, {}).setdefault(label, {})
        for seed in SEEDS:
            set_seed(seed)
            model = SGBAControl(
                bias_type=bias_type, pe_type=None,
                elev_mean=norm_stats['elev_mean'],
                elev_std=norm_stats['elev_std'],
            )
            ckpt = OUTPUT_CKPTS / f"ablation_{site}_{label}_seed{seed}.pt"
            res = train_sgba_control(model, train_loader, val_loader,
                                     epochs=EPOCHS, lr=LR, weight_decay=WD,
                                     grad_clip=GRAD_CLIP, device=DEVICE,
                                     checkpoint_path=ckpt,
                                     verbose=False, exp_id=f"abl_{site}_{label}_{seed}")
            ev = evaluate_sgba_control(res['model'], test_loader,
                                       device=DEVICE, site_key=site,
                                       data_root=DATA_ROOT)
            m = ev['metrics']
            ablation_results[site][label][seed] = m
            print(f"  {label:20s} seed={seed:3d}  "
                  f"RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}  R2={m['R2']:.4f}")

with open(OUTPUT_METRICS / 'ablation_summary.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

# Summary table
print("\nAblation summary (mean across seeds, RMSE / R2)")
print(f"{'Site':<10}{'SGBA-Elev RMSE':<20}{'Shuffled RMSE':<20}{'dRMSE':<12}{'Elev R2':<10}")
for site in ABLATION_SITES:
    e = ablation_results[site]['SGBA-Elev']
    s = ablation_results[site]['SGBA-ShuffledElev']
    e_rmse = np.mean([e[k]['RMSE'] for k in e])
    s_rmse = np.mean([s[k]['RMSE'] for k in s])
    e_r2 = np.mean([e[k]['R2'] for k in e])
    print(f"{site:<10}{e_rmse:<20.4f}{s_rmse:<20.4f}{e_rmse - s_rmse:<+12.4f}{e_r2:<10.4f}")


## 7. Run Multi-Horizon Experiments

SGBA-Elev vs PatchTST at h in {1, 6, 12, 24}.

In [ ]:
import json
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader

MH_SITES = ['site7', 'site10', 'site2']
MH_HORIZONS = [1, 6, 12, 24]
DATA_ROOT = DKASC_DIR.parent

mh_results = {}

for site in MH_SITES:
    # Load raw arrays directly so we can re-build targets per horizon.
    site_dir = DKASC_DIR / site
    X_tr = np.load(site_dir / 'X_train.npy')
    X_va = np.load(site_dir / 'X_val.npy')
    X_te = np.load(site_dir / 'X_test.npy')
    y_tr = np.load(site_dir / 'y_train.npy')
    y_va = np.load(site_dir / 'y_val.npy')
    y_te = np.load(site_dir / 'y_test.npy')
    h_tr = np.load(site_dir / 'hour_train.npy')
    h_va = np.load(site_dir / 'hour_val.npy')
    h_te = np.load(site_dir / 'hour_test.npy')
    e_tr = np.load(site_dir / 'elev_train.npy')
    e_va = np.load(site_dir / 'elev_val.npy')
    e_te = np.load(site_dir / 'elev_test.npy')
    ts_tr = np.load(site_dir / 'ts_train.npy', allow_pickle=True)
    ts_te = np.load(site_dir / 'ts_test.npy', allow_pickle=True)

    # Get normalization stats once.
    _, _, _, norm_stats = create_dataloaders_v2(site, DATA_ROOT, batch_size=BATCH_SIZE)

    for h in MH_HORIZONS:
        y_h_tr, m_tr = build_multihorizon_targets(ts_tr, y_tr, h)
        y_h_te, m_te = build_multihorizon_targets(ts_te, y_te, h)

        X_tr_h, y_tr_h, h_tr_h, e_tr_h = X_tr[m_tr], y_h_tr[m_tr], h_tr[m_tr], e_tr[m_tr]
        X_te_h, y_te_h, h_te_h, e_te_h = X_te[m_te], y_h_te[m_te], h_te[m_te], e_te[m_te]
        # Use last 15% of training as validation (chronological split).
        n_val = int(0.15 * len(X_tr_h))
        X_va_h, y_va_h, h_va_h, e_va_h = X_tr_h[-n_val:], y_tr_h[-n_val:], h_tr_h[-n_val:], e_tr_h[-n_val:]
        X_tr_h2, y_tr_h2, h_tr_h2, e_tr_h2 = X_tr_h[:-n_val], y_tr_h[:-n_val], h_tr_h[:-n_val], e_tr_h[:-n_val]

        train_ds = MultiHorizonDataset(X_tr_h2, y_tr_h2, h_tr_h2, e_tr_h2)
        val_ds = MultiHorizonDataset(X_va_h, y_va_h, h_va_h, e_va_h)
        test_ds = MultiHorizonDataset(X_te_h, y_te_h, h_te_h, e_te_h)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

        mh_results.setdefault(site, {}).setdefault(h, {})
        for name, mk in [
            ('SGBA-Elev', lambda: SGBA(geometry_mode='elev',
                                       elev_mean=norm_stats['elev_mean'],
                                       elev_std=norm_stats['elev_std'])),
            ('PatchTST', lambda: PatchTST()),
        ]:
            set_seed(42)
            model = mk()
            use_geo = name == 'SGBA-Elev'
            model = train_multihorizon(model, train_loader, val_loader,
                                       use_geometry=use_geo, epochs=EPOCHS,
                                       lr=LR, weight_decay=WD, grad_clip=GRAD_CLIP,
                                       device=DEVICE, verbose=False)
            ev = evaluate_multihorizon(model, test_loader, use_geometry=use_geo,
                                        device=DEVICE, site_key=site, data_root=DATA_ROOT)
            m = ev['metrics']
            mh_results[site][h][name] = m
            print(f"  {site} h={h:2d}  {name:10s}  RMSE={m['RMSE']:.4f}  R2={m['R2']:.4f}")

with open(OUTPUT_METRICS / 'multihorizon_summary.json', 'w') as f:
    json.dump(mh_results, f, indent=2)

# Summary table
print("\nMulti-horizon summary (mean RMSE across 3 sites)")
print(f"{'Horizon':<10}{'SGBA-Elev RMSE':<20}{'PatchTST RMSE':<20}{'dRMSE':<12}")
for h in MH_HORIZONS:
    s_vals = [mh_results[s][h]['SGBA-Elev']['RMSE'] for s in MH_SITES]
    p_vals = [mh_results[s][h]['PatchTST']['RMSE'] for s in MH_SITES]
    s_mean = np.mean(s_vals)
    p_mean = np.mean(p_vals)
    print(f"h={h:<8}{s_mean:<20.4f}{p_mean:<20.4f}{s_mean - p_mean:<+12.4f}")


## 8. Efficiency Measurement

GPU latency, FLOPs (thop), and ONNX export with CPU latency.

In [ ]:
import time
import json
import numpy as np
import torch


def count_flops_thop(model, sample_input):
    """FLOPs via thop. Returns None if thop not installed."""
    try:
        from thop import profile
        flops, params = profile(model, inputs=sample_input, verbose=False)
        return int(flops)
    except Exception:
        return None


def measure_gpu_latency(model, x, hour, elev, n_warmup=10, n_trials=100):
    """Single-sample GPU inference latency in ms."""
    model.eval()
    device = next(model.parameters()).device
    x = x.to(device); hour = hour.to(device); elev = elev.to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            try:
                model(x, hour, elev)
            except TypeError:
                model(x, hour)
        if device.type == 'cuda':
            torch.cuda.synchronize()
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            for _ in range(n_trials):
                try:
                    model(x, hour, elev)
                except TypeError:
                    model(x, hour)
            end.record()
            torch.cuda.synchronize()
            return start.elapsed_time(end) / n_trials
        else:
            t0 = time.time()
            for _ in range(n_trials):
                try:
                    model(x, hour, elev)
                except TypeError:
                    model(x, hour)
            return (time.time() - t0) / n_trials * 1000


def measure_onnx_cpu_latency(model, x, hour, elev, n_trials=100):
    """Export to ONNX and measure CPU latency with onnxruntime."""
    try:
        import onnx
        import onnxruntime as ort
    except ImportError:
        return None, None
    model.eval()
    onnx_path = OUTPUT_ROOT / f"{model.__class__.__name__}.onnx"
    try:
        torch.onnx.export(model, (x, hour, elev), str(onnx_path),
                          input_names=['x', 'hour', 'elev'],
                          output_names=['y'], opset_version=14)
    except Exception:
        try:
            torch.onnx.export(model, (x,), str(onnx_path),
                              input_names=['x'], output_names=['y'], opset_version=14)
        except Exception:
            return None, None
    sess = ort.InferenceSession(str(onnx_path))
    in_names = [i.name for i in sess.get_inputs()]
    feed = {}
    if 'x' in in_names: feed['x'] = x.numpy().astype(np.float32)
    if 'hour' in in_names: feed['hour'] = hour.numpy().astype(np.float32)
    if 'elev' in in_names: feed['elev'] = elev.numpy().astype(np.float32)
    for _ in range(10): sess.run(None, feed)
    t0 = time.time()
    for _ in range(n_trials): sess.run(None, feed)
    return (time.time() - t0) / n_trials * 1000, str(onnx_path)


B, L, C = 1, 24, 5
x = torch.randn(B, L, C)
hour = torch.rand(B, L) * 24
elev = torch.rand(B, L) * 60

eff_models = {
    'PatchTST': PatchTST(),
    'SimpleMLP': SimpleMLP(),
    'SimpleLSTM': SimpleLSTM(),
    'SimpleGRU': SimpleGRU(),
    'CTNet': CTNet(),
    'SGBA-Elev': SGBA(geometry_mode='elev'),
    'SGBA-ShuffledElev': SGBAControl(bias_type='shuffled'),
}

eff_results = {}
for name, model in eff_models.items():
    model = model.to(DEVICE)
    params = model.count_parameters()
    x_d = x.to(DEVICE); hour_d = hour.to(DEVICE); elev_d = elev.to(DEVICE)
    flops = count_flops_thop(model, (x_d, hour_d, elev_d))
    gpu_lat = measure_gpu_latency(model, x_d, hour_d, elev_d)
    onnx_lat, _ = measure_onnx_cpu_latency(model, x.cpu(), hour.cpu(), elev.cpu())
    eff_results[name] = {'params': params, 'flops': flops,
                         'gpu_latency_ms': gpu_lat, 'onnx_cpu_latency_ms': onnx_lat,
                         'device': str(DEVICE)}
    onnx_str = f"{onnx_lat:.3f}ms" if onnx_lat is not None else "N/A"
    print(f"  {name:20s} params={params:>8,}  flops={flops}  "
          f"gpu={gpu_lat:.3f}ms  onnx_cpu={onnx_str}")

with open(OUTPUT_METRICS / 'efficiency_summary.json', 'w') as f:
    json.dump(eff_results, f, indent=2, default=str)
